# 🌌 Windowed Dual-Space Centroid KV: Flagship 7B/8B LLM Evaluation
### End-to-End Generative Evaluation on Qwen2.5-7B-Instruct and Llama-3.1-8B-Instruct

This Kaggle notebook evaluates **Phase-Coherent Windowed Dual-Space Centroid KV** against **H2O** and **StreamingLLM** on large 7B/8B language models across 4,096+ tokens under strict byte-for-byte cache budgets and Perplexity (PPL) scaling.

> ⚙️ **Kaggle Setup:**
> 1. Go to **Settings** (right sidebar).
> 2. Set **Accelerator** to **GPU T4 x2** (or GPU T4).
> 3. Set **Internet** to **ON** (required to download model weights from Hugging Face).
> 4. Click **Run All**.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB VRAM)")

In [ ]:
# 2. Clone Repository & Install Dependencies
!pip install -q transformers accelerate bitsandbytes matplotlib
%cd /kaggle/working
!rm -rf dual-space-kv
!git clone https://github.com/just-shutup/dual-space-kv.git
%cd /kaggle/working/dual-space-kv
!pip install -q -e .

In [ ]:
# 3. Flagship Model 1: Qwen2.5-7B-Instruct (4,096 tokens, 32x compression, 4-bit weights)
%cd /kaggle/working/dual-space-kv
!python benchmarks/run_7b_evaluation.py --model_id Qwen/Qwen2.5-7B-Instruct --context_len 4096 --budget 128 --load_in_4bit

In [ ]:
# 4. Flagship Model 2: Meta-Llama-3.1-8B-Instruct (4,096 tokens, 32x compression, 4-bit weights)
%cd /kaggle/working/dual-space-kv
!python benchmarks/run_7b_evaluation.py --model_id NousResearch/Meta-Llama-3.1-8B-Instruct --context_len 4096 --budget 128 --load_in_4bit

In [ ]:
# 5. Long-Context Perplexity (PPL) Scaling Benchmark (512 -> 4096 tokens, 4x compression)
%cd /kaggle/working/dual-space-kv
!python benchmarks/run_ppl_benchmark.py --model_id Qwen/Qwen2.5-7B-Instruct --context_lengths '512,1024,2048,4096' --eval_tokens 64 --compression_ratio 4 --load_in_4bit

# Display Generated Perplexity Curve Plot
from IPython.display import Image, display
display(Image('/kaggle/working/dual-space-kv/paper/ppl_scaling_curve.png'))

In [ ]:
# 6. Display All Benchmark Results Summaries
import json
import glob

res_files = sorted(glob.glob("/kaggle/working/dual-space-kv/paper/results_*.json"))
for fpath in res_files:
    with open(fpath) as f:
        data = json.load(f)
    print("\n" + "="*85)
    print(f"RESULTS: {fpath}")
    print("="*85)
    for k, v in data.items():
        status = "✅ 100% ACCURACY 🏆" if v['acc'] else "❌ 0% HALLUCINATION"
        print(f"{k:<25} | Budget: {v['budget']:<5} | Output: {v['output']:<22} | {status}")
    print("="*85)

ppl_files = sorted(glob.glob("/kaggle/working/dual-space-kv/paper/ppl_results_*.json"))
for fpath in ppl_files:
    with open(fpath) as f:
        pdata = json.load(f)
    print("\n" + "="*85)
    print(f"PERPLEXITY (PPL) SCALING: {pdata['model_id']}")
    print("="*85)
    print(f"{'Context':<10} | {'Exact PPL':<12} | {'StreamingLLM':<14} | {'H2O':<12} | {'DualSpace':<12}")
    print("-"*85)
    for i, l in enumerate(pdata['lengths']):
        if i < len(pdata['exact']):
            print(f"{l:<10} | {pdata['exact'][i]:<12.3f} | {pdata['streaming_llm'][i]:<14.3f} | {pdata['h2o'][i]:<12.3f} | {pdata['dual_space'][i]:<12.3f}")
    print("="*85)